# Gate 2 · e07 — progressive parameter reduction of the 3 best CNNs (Colab GPU + Weights & Biases)

Takes the **three models with the best macro-F1 on the 2025 fold** of `cv_per_fold.csv` (`RANK_FOLD = "2025"`; in e06: `efficientnet_b0`, `crnn`, `cnn_deep`), scales each one to **≈ 1 M trainable parameters**
(the FPGA budget of `full_model`) and then **reduces the width step by step** (1.0 → 0.7 → 0.5 → 0.35 → 0.25 → 0.15 → 0.08 M). For this test every size is retrained **only on the 2025 fold**
(train 2022–2024, test 2025; validation = real patients inside the training years, the held-out year never takes part in training or selection). Presets `long` / `full` use the three folds 2023–2025.

For every training it logs a **Weights & Biases run** (loss / validation AUC / validation macro-F1 per epoch, held-out metrics and confusion matrix); at the end it writes CSV reports
(`report_per_run.csv`, `confusion_matrices.csv`, `report_summary.csv` and, last, `report_final.csv` with the recommended size of each model) and a W&B summary run with the comparison charts.
**It needs a CUDA GPU** (Runtime → Change runtime type → T4) and stops with an error otherwise. Same data, mel recipe, augmentation and training recipe as e06.

Run it from VS Code (Colab extension): *Select Kernel → Colab → GPU*. W&B (online, your account): store your key (https://wandb.ai/authorize) as a Colab secret `WANDB_API_KEY`, or paste it into the hidden prompt of the setup cell; optionally set `WANDB_ENTITY`. Start with `PRESET = "smoke"`.

In [ ]:
# ── Options ──────────────────────────────────────────────────────────────────────────────────────────
from pathlib import Path

PRESET = "fold2025"     # "smoke" (minutes) -> "fold2025" (this test: only the 2025 fold) -> "long" / "full" (3 folds)
TOP_K = 3               # number of models taken from cv_per_fold.csv
RANK_FOLD = "2025"      # rank the models by their macro_f1 on this fold only ("2025" for this test); None = mean over all folds
CV_CSV = Path("/content/cv_per_fold.csv")   # optional: upload e06's cv_per_fold.csv here; otherwise the ranking below (computed from it) is used
FALLBACK_MACRO_F1 = {   # macro_f1 per model, condition "real" (from e06 cv_per_fold.csv): fold 2025, and the mean over the three folds
    "2025": {"efficientnet_b0": 0.746287, "crnn": 0.736953, "cnn_deep": 0.734557, "resnet_small": 0.730447,
             "cnn_baseline": 0.682236, "resnet18": 0.554138, "cnn_se": 0.514001, "cnn_wide": 0.320843},
    None: {"cnn_deep": 0.838920, "crnn": 0.834283, "resnet_small": 0.828828, "cnn_baseline": 0.797743,
           "efficientnet_b0": 0.790153, "resnet18": 0.758625, "cnn_se": 0.756520, "cnn_wide": 0.690696}}

TARGETS_M = [1.0, 0.7, 0.5, 0.35, 0.25, 0.15, 0.08]    # trainable parameters (millions) of each step, from the budget downwards
PRESETS = {
    "smoke": dict(FOLDS=["2025"], SEEDS=[0], MAX_EPOCHS=3, LEVELS=[1.0, 0.25]),
    "fold2025": dict(FOLDS=["2025"], SEEDS=[0, 1], MAX_EPOCHS=30, LEVELS=TARGETS_M),      # this test: held-out year 2025 only
    "long": dict(FOLDS=["2023", "2024", "2025"], SEEDS=[0, 1], MAX_EPOCHS=30, LEVELS=TARGETS_M),
    "full": dict(FOLDS=["2023", "2024", "2025"], SEEDS=[0, 1, 2], MAX_EPOCHS=40, LEVELS=TARGETS_M),
}
FOLDS, SEEDS, MAX_EPOCHS, LEVELS = (PRESETS[PRESET][k] for k in ("FOLDS", "SEEDS", "MAX_EPOCHS", "LEVELS"))
OFFICIAL_FOLD = "2025"
TIME_BUDGET_MIN = 240                    # the loop stops cleanly before exceeding this (keep it below your Colab session limit)
TOLERANCE = 0.01                         # recommended size = smallest one within this much macro-F1 of the model's best size

WANDB_PROJECT, WANDB_MODE = "gate2-param-reduction", "online"    # "online" | "offline" | "disabled"
WANDB_ENTITY = ""                        # your W&B username or team; empty = the default entity of the account whose key you use
REQUIRE_GPU = True                       # stop if CUDA is not available (set False only for a CPU test)

DATA_ROOT = Path("/content/data/raw_data")
OUT_DIR = Path("/content/outputs")
PATIENCE, BATCH = 5, 64
LR, WEIGHT_DECAY = 3e-4, 1e-3
THRESHOLD = 0.5

In [ ]:
# ── Setup + GPU check ────────────────────────────────────────────────────────────────────────────────
import subprocess, sys
subprocess.run([sys.executable, "-m", "pip", "install", "-q", "soundfile", "wandb"], check=False)

import json, os, shutil, time, warnings
from math import gcd
import numpy as np
import pandas as pd
import torch, torch.nn as nn
import matplotlib
import matplotlib.pyplot as plt
import soundfile as sf
from joblib import Parallel, delayed
from scipy import signal
from sklearn.metrics import classification_report, confusion_matrix, roc_auc_score, balanced_accuracy_score, f1_score
from sklearn.model_selection import GroupShuffleSplit

warnings.filterwarnings("ignore")
if REQUIRE_GPU and not torch.cuda.is_available():
    raise RuntimeError("No CUDA GPU: Runtime > Change runtime type > Hardware accelerator = GPU (T4), then reconnect and run again.")
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
torch.backends.cudnn.benchmark = True
print("device:", DEVICE, torch.cuda.get_device_name(0) if DEVICE.type == "cuda" else "(CPU: test mode)")
if DEVICE.type == "cuda":
    print(f"GPU memory: {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB")
OUT_DIR.mkdir(parents=True, exist_ok=True)

# Weights & Biases login with YOUR account (online). Key from: env WANDB_API_KEY -> Colab secret WANDB_API_KEY -> a hidden prompt.
import wandb
os.environ["WANDB_MODE"] = WANDB_MODE
os.environ["WANDB_SILENT"] = "true"
if WANDB_MODE == "online":
    key = os.environ.get("WANDB_API_KEY")
    if not key:
        try:
            from google.colab import userdata
            key = userdata.get("WANDB_API_KEY")
        except Exception:
            key = None
    if not key:
        from getpass import getpass
        key = getpass("Paste your W&B API key (https://wandb.ai/authorize), it is not echoed: ").strip()
    os.environ["WANDB_API_KEY"] = key
    wandb.login(key=key, verify=True)                       # raises if the key is wrong: the run never silently goes offline
    try:
        print("W&B logged in as:", wandb.Api().viewer.username, "| project:", f"{WANDB_ENTITY or '<default entity>'}/{WANDB_PROJECT}")
    except Exception:
        print("W&B login verified | project:", f"{WANDB_ENTITY or '<default entity>'}/{WANDB_PROJECT}")
print(f"preset {PRESET}: folds {FOLDS}, seeds {SEEDS}, levels {LEVELS} M params, max {MAX_EPOCHS} epochs, budget {TIME_BUDGET_MIN} min, wandb {WANDB_MODE}")

## 1 · Data: download SPRSound (train 2022–2024, test 2025) and extract the W / FC events as log-mel

In [ ]:
import shutil

def download_sprsound(root: Path):
    """Clone SPRSound and copy the recordings to root/{train,test}/{wav,json}, prefixing the file names with the year (as get_data.sh)."""
    work = Path("/content/SPRSound")
    if not work.exists():
        subprocess.run(["git", "clone", "--depth", "1", "https://github.com/SJTU-YONGFU-RESEARCH-GRP/SPRSound.git", str(work)], check=True)
    sources = [("BioCAS2022/train2022_wav", "BioCAS2022/train2022_json", "train", "2022"),
               ("BioCAS2022/test2022_wav", "BioCAS2022/test2022_json", "train", "2022"),
               ("BioCAS2023/test2023_wav", "BioCAS2023/test2023_json", "train", "2023"),
               ("BioCAS2024/test2024_wav", "BioCAS2024/test2024_json", "train", "2024"),
               ("BioCAS2025/test2025_wav", "BioCAS2025/test2025_json", "test", "2025")]
    for wav_dir, json_dir, split, year in sources:
        if not (work / wav_dir).is_dir():
            print("not found, skipped:", wav_dir)
            continue
        (root / split / "wav").mkdir(parents=True, exist_ok=True)
        (root / split / "json").mkdir(parents=True, exist_ok=True)
        jsons = {p.name: p for p in (work / json_dir).rglob("*.json")}
        n = 0
        for wav in sorted((work / wav_dir).glob("*.wav")):
            js = jsons.get(wav.stem + ".json")
            if js is None:
                continue
            shutil.copy2(wav, root / split / "wav" / f"{year}__{wav.name}")
            shutil.copy2(js, root / split / "json" / f"{year}__{js.name}")
            n += 1
        print(f"{split:5s} {year}: {n} recordings")

if not list((DATA_ROOT / "train" / "wav").glob("*.wav")):
    download_sprsound(DATA_ROOT)
print({s: len(list((DATA_ROOT / s / "wav").glob("*.wav"))) for s in ("train", "test")})

In [ ]:
SR, WIN_S, N_FFT, HOP, N_MELS, FMIN, FMAX = 4000, 3.0, 512, 128, 64, 50.0, 2000.0
CLASSES = ["W", "FC"]
EVENT_TYPES = {"wheeze": 0, "fine crackle": 1}

def mel_filterbank():
    freqs = np.fft.rfftfreq(N_FFT, 1 / SR)
    mel = lambda f: 2595 * np.log10(1 + f / 700)
    inv = lambda m: 700 * (10 ** (m / 2595) - 1)
    pts = inv(np.linspace(mel(FMIN), mel(FMAX), N_MELS + 2))
    fb = np.zeros((N_MELS, len(freqs)))
    for i in range(N_MELS):
        lo, c, hi = pts[i], pts[i + 1], pts[i + 2]
        fb[i] = np.maximum(0, np.minimum((freqs - lo) / (c - lo), (hi - freqs) / (hi - c)))
    return fb

FB = mel_filterbank()

def to_mel(seg):
    n = int(WIN_S * SR)
    w = seg[(len(seg) - n) // 2:(len(seg) - n) // 2 + n] if len(seg) >= n else np.tile(seg, int(np.ceil(n / len(seg))))[:n]
    _, _, z = signal.stft(w, nperseg=N_FFT, noverlap=N_FFT - HOP, window="hann", boundary="even", padded=True)
    d = 10 * np.log10(np.maximum(FB @ (np.abs(z) ** 2), 1e-10))
    return ((d - d.mean()) / (d.std() + 1e-6)).astype(np.float32)

def recording_events(wav, js, year, patient):
    ann = json.loads(Path(js).read_text(encoding="utf-8"))
    ev = []
    for e in ann.get("event_annotation", []):
        label = EVENT_TYPES.get(str(e.get("type", "")).strip().lower())
        try:
            s, t = int(e["start"]), int(e["end"])
        except (KeyError, ValueError):
            continue
        if label is not None and t - s >= 20:
            ev.append((label, s, t))
    if not ev:
        return []
    x, sr = sf.read(str(wav), dtype="float32", always_2d=True)
    x = x[:, 0]
    if sr != SR:
        g = gcd(int(sr), SR)
        x = signal.resample_poly(x, SR // g, sr // g).astype(np.float32)
    out = []
    for label, s, t in ev:
        seg = x[int(s * SR / 1000):int(t * SR / 1000)]
        if len(seg) >= 64:
            out.append((to_mel(seg), label, year, patient))
    return out

def load_real_events(root):
    jobs = []
    for split in ("train", "test"):
        for wav in sorted((root / split / "wav").glob("*.wav")):
            js = root / split / "json" / f"{wav.stem}.json"
            year, rest = wav.stem.split("__", 1)
            pid = rest.split("_")[0]
            if js.exists():
                jobs.append((wav, js, year, str(float(pid)) if pid.isdigit() else "nan"))
    rows = [r for rs in Parallel(n_jobs=-1)(delayed(recording_events)(*j) for j in jobs) for r in rs]
    X = np.stack([r[0] for r in rows])
    y, year, patient = (np.array([r[i] for r in rows]) for i in (1, 2, 3))
    return X, y, year, patient

cache = OUT_DIR / "real_events.npz"
if cache.exists():
    z = np.load(cache, allow_pickle=True); X, y, year, patient = z["X"], z["y"], z["year"], z["patient"]
else:
    X, y, year, patient = load_real_events(DATA_ROOT)
    np.savez(cache, X=X, y=y, year=year, patient=patient)
tab = pd.crosstab(year, np.array(CLASSES)[y]).rename_axis("year")
print(f"{len(y)} events"); display(tab)

## 2 · Which models? The top-K by mean macro-F1 of `cv_per_fold.csv`

In [ ]:
if CV_CSV.exists():
    cv_in = pd.read_csv(CV_CSV)
    cv_in = cv_in[cv_in["condition"] == "real"] if "real" in set(cv_in["condition"]) else cv_in
    if RANK_FOLD is not None:
        cv_in = cv_in[cv_in["fold"].astype(str) == str(RANK_FOLD)]
    ranking = cv_in.groupby("model")["macro_f1"].mean().sort_values(ascending=False)
    print("ranking computed from", CV_CSV)
else:
    ranking = pd.Series(FALLBACK_MACRO_F1[RANK_FOLD]).sort_values(ascending=False)
    print("ranking from the values embedded above (upload cv_per_fold.csv to", CV_CSV, "to recompute it)")
print(f"macro_f1 used for the ranking: {'fold ' + str(RANK_FOLD) if RANK_FOLD else 'mean over all folds'}")
MODELS = ranking.index[:TOP_K].tolist()
display(ranking.rename("macro_f1").to_frame().assign(selected=lambda d: d.index.isin(MODELS)).round(4))
print("selected:", MODELS)

## 3 · Models and width scaling

The architectures of e06 with a **width multiplier** `w` applied to every channel count (rounded to a multiple of 4; for `crnn` also the GRU size). For each target size the multiplier whose
parameter count is closest to the target is searched, so every step is a real, trainable network of ≈ that many parameters.

In [ ]:
import torchvision

def conv_bn(c_in, c_out):
    return [nn.Conv2d(c_in, c_out, 3, padding=1, bias=False), nn.BatchNorm2d(c_out), nn.ReLU(inplace=True)]

class SE(nn.Module):
    def __init__(self, c, r=4):
        super().__init__()
        self.fc = nn.Sequential(nn.Linear(c, max(c // r, 4)), nn.ReLU(), nn.Linear(max(c // r, 4), c), nn.Sigmoid())
    def forward(self, x):
        return x * self.fc(x.mean((2, 3)))[:, :, None, None]

def head(c, drop=0.3):
    return nn.Sequential(nn.Dropout(drop), nn.Linear(c, 64), nn.ReLU(), nn.Linear(64, 1))

class PlainCNN(nn.Module):
    """Blocks of `reps` × (Conv3×3-BN-ReLU) + optional SE + MaxPool, then global average pooling and an MLP head."""
    def __init__(self, ch=(16, 32, 64, 128), reps=1, se=False):
        super().__init__()
        layers, c_in = [], 1
        for c in ch:
            for r in range(reps):
                layers += conv_bn(c_in if r == 0 else c, c)
            layers += ([SE(c)] if se else []) + [nn.MaxPool2d(2)]
            c_in = c
        self.conv, self.head = nn.Sequential(*layers), head(ch[-1])
    def forward(self, x):
        return self.head(self.conv(x).mean((2, 3))).squeeze(1)

class BasicBlock(nn.Module):
    def __init__(self, c_in, c_out, stride):
        super().__init__()
        self.body = nn.Sequential(nn.Conv2d(c_in, c_out, 3, stride, 1, bias=False), nn.BatchNorm2d(c_out), nn.ReLU(inplace=True),
                                  nn.Conv2d(c_out, c_out, 3, 1, 1, bias=False), nn.BatchNorm2d(c_out))
        self.skip = nn.Identity() if stride == 1 and c_in == c_out else nn.Sequential(nn.Conv2d(c_in, c_out, 1, stride, bias=False), nn.BatchNorm2d(c_out))
    def forward(self, x):
        return torch.relu(self.body(x) + self.skip(x))

class ResNetSmall(nn.Module):
    def __init__(self, ch=(16, 32, 64, 128)):
        super().__init__()
        layers, c_in = [nn.Sequential(*conv_bn(1, ch[0]))], ch[0]
        for i, c in enumerate(ch):
            layers.append(BasicBlock(c_in, c, 1 if i == 0 else 2)); c_in = c
        self.body, self.head = nn.Sequential(*layers), head(ch[-1])
    def forward(self, x):
        return self.head(self.body(x).mean((2, 3))).squeeze(1)

class CRNN(nn.Module):
    def __init__(self, ch=(16, 32, 64), hidden=96):
        super().__init__()
        layers, c_in = [], 1
        for i, c in enumerate(ch):
            layers += conv_bn(c_in, c) + [nn.MaxPool2d(2 if i < 2 else (2, 1))]
            c_in = c
        self.conv = nn.Sequential(*layers)
        self.rnn = nn.GRU(ch[-1] * (N_MELS // 8), hidden, batch_first=True, bidirectional=True)
        self.head = head(2 * hidden)
    def forward(self, x):
        h = self.conv(x)                                   # (B, C, F', T')
        h = h.permute(0, 3, 1, 2).flatten(2)               # (B, T', C·F')
        return self.head(self.rnn(h)[0].mean(1)).squeeze(1)

def make_resnet18():
    m = torchvision.models.resnet18(weights=None, num_classes=1)
    m.conv1 = nn.Conv2d(1, 64, 3, 1, 1, bias=False); m.maxpool = nn.Identity()    # small input: keep resolution
    return Squeeze(m)

def make_efficientnet_b0():
    m = torchvision.models.efficientnet_b0(weights=None, num_classes=1)
    m.features[0][0] = nn.Conv2d(1, 32, 3, 1, 1, bias=False)
    return Squeeze(m)

class Squeeze(nn.Module):
    def __init__(self, m):
        super().__init__(); self.m = m
    def forward(self, x):
        return self.m(x).squeeze(1)

In [ ]:
BASE = {"cnn_deep": (16, 32, 64, 128, 128), "resnet_small": (16, 32, 64, 128), "crnn": (16, 32, 64), "efficientnet_b0": (32, 16, 24, 40, 80, 112, 192, 320)}

from functools import partial
from torchvision.models.efficientnet import EfficientNet, MBConvConfig

def make_efficientnet(w):
    """torchvision EfficientNet-B0 layout with width multiplier w (w = 1.0 is the 4.0 M-parameter e06 model), 1 input channel, stride-1 stem."""
    b = partial(MBConvConfig, width_mult=w, depth_mult=1.0)
    setting = [b(1, 3, 1, 32, 16, 1), b(6, 3, 2, 16, 24, 2), b(6, 5, 2, 24, 40, 2), b(6, 3, 2, 40, 80, 3), b(6, 5, 1, 80, 112, 3), b(6, 5, 2, 112, 192, 4), b(6, 3, 1, 192, 320, 1)]
    m = EfficientNet(setting, dropout=0.2, num_classes=1)
    m.features[0][0] = nn.Conv2d(1, m.features[0][0].out_channels, 3, 1, 1, bias=False)
    return Squeeze(m)

def make_model(name, w=1.0):
    if name == "efficientnet_b0":
        return make_efficientnet(w)
    r = lambda c: max(4, int(round(c * w / 4)) * 4)
    ch = tuple(r(c) for c in BASE[name])
    if name == "cnn_deep":
        return PlainCNN(ch, reps=2)
    if name == "resnet_small":
        return ResNetSmall(ch)
    if name == "crnn":
        return CRNN(ch, hidden=r(96))
    raise ValueError(f"{name} is not scalable here (add it to BASE / make_model)")

def n_params(name, w):
    return sum(p.numel() for p in make_model(name, w).parameters())

GRID = np.round(np.geomspace(0.05, 4.0, 500), 4)
WIDTHS, rows = {}, []
for name in MODELS:
    counts = np.array([n_params(name, w) for w in GRID])
    for k, target in enumerate(LEVELS):
        ok = np.flatnonzero(counts <= target * 1e6)          # largest size that does not exceed the target (so the 1.0 M step is < 1 M)
        i = int(ok[np.argmax(counts[ok])]) if len(ok) else int(np.argmin(np.abs(counts - target * 1e6)))
        WIDTHS[(name, k)] = float(GRID[i])
        rows.append(dict(model=name, step=k, target_M=target, width_mult=float(GRID[i]), params=int(counts[i]), channels=("EfficientNet-B0 layout, width x%.3f" % GRID[i]) if name == "efficientnet_b0" else str(tuple(max(4, int(round(c * GRID[i] / 4)) * 4) for c in BASE[name]))))
size_table = pd.DataFrame(rows)
display(size_table)
for name in MODELS:                                      # forward-shape check on every size
    for k in range(len(LEVELS)):
        assert make_model(name, WIDTHS[(name, k)])(torch.zeros(2, 1, 64, 95)).shape == (2,)

## 4 · Training and evaluation (real events only, GPU, W&B run per training)

In [ ]:
def augment(x):
    """GPU SpecAugment-like: circular time shift, one frequency mask (≤ 8 bins), one time mask (≤ 10 frames), per sample."""
    B, _, F, T = x.shape
    dev = x.device
    shift = torch.randint(0, T, (B,), device=dev)
    idx = (torch.arange(T, device=dev)[None] - shift[:, None]) % T
    x = x.gather(3, idx[:, None, None, :].expand(B, 1, F, T))
    f = torch.randint(0, 9, (B,), device=dev); f0 = (torch.rand(B, device=dev) * (F - f)).long()
    ar = torch.arange(F, device=dev)[None]
    x = x.masked_fill(((ar >= f0[:, None]) & (ar < (f0 + f)[:, None]))[:, None, :, None], 0)
    t = torch.randint(0, 11, (B,), device=dev); t0 = (torch.rand(B, device=dev) * (T - t)).long()
    ar = torch.arange(T, device=dev)[None]
    return x.masked_fill(((ar >= t0[:, None]) & (ar < (t0 + t)[:, None]))[:, None, None, :], 0)

@torch.no_grad()
def predict(model, Xg, bs=512):
    model.eval()
    out = []
    for i in range(0, len(Xg), bs):
        with torch.autocast(DEVICE.type, enabled=DEVICE.type == "cuda"):
            out.append(torch.sigmoid(model(Xg[i:i + bs]).float()))
    return torch.cat(out).cpu().numpy()

In [ ]:
def run_epochs(model, opt, loss_fn, Xg, yg, n_epochs, scaler, on_epoch=None):
    for ep in range(n_epochs):
        model.train()
        perm = torch.randperm(len(Xg), device=DEVICE)
        tot, nb = 0.0, 0
        for i in range(0, len(perm), BATCH):
            b = perm[i:i + BATCH]
            if len(b) < 4:
                continue
            with torch.autocast(DEVICE.type, enabled=DEVICE.type == "cuda"):
                loss = loss_fn(model(augment(Xg[b])).float(), yg[b])
            opt.zero_grad(set_to_none=True)
            scaler.scale(loss).backward(); scaler.step(opt); scaler.update()
            tot += float(loss.detach()); nb += 1
        if on_epoch is not None and on_epoch(ep, tot / max(nb, 1)):
            break

def metrics_of(y_true, p, thr=THRESHOLD):
    pred = (p >= thr).astype(int)
    r = classification_report(y_true, pred, labels=[0, 1], target_names=CLASSES, output_dict=True, zero_division=0)
    tn, fp, fn, tp = confusion_matrix(y_true, pred, labels=[0, 1]).ravel()
    return {"test_auc": roc_auc_score(y_true, p), "balanced_acc": balanced_accuracy_score(y_true, pred), "accuracy": r["accuracy"],
            "precision_W": r["W"]["precision"], "recall_W": r["W"]["recall"], "f1_W": r["W"]["f1-score"],
            "precision_FC": r["FC"]["precision"], "recall_FC": r["FC"]["recall"], "f1_FC": r["FC"]["f1-score"],
            "macro_f1": r["macro avg"]["f1-score"], "weighted_f1": r["weighted avg"]["f1-score"],
            "tn_W_as_W": int(tn), "fp_W_as_FC": int(fp), "fn_FC_as_W": int(fn), "tp_FC_as_FC": int(tp)}

def cm_figure(y_true, p, title):
    cm = confusion_matrix(y_true, (p >= THRESHOLD).astype(int), labels=[0, 1])
    cmn = cm / cm.sum(1, keepdims=True)
    fig, ax = plt.subplots(figsize=(3.6, 3.4))
    ax.imshow(cmn, vmin=0, vmax=1, cmap="Blues")
    for i in range(2):
        for j in range(2):
            ax.text(j, i, f"{cm[i, j]}\n{cmn[i, j]:.1%}", ha="center", va="center", color="white" if cmn[i, j] > .5 else "black")
    ax.set_xticks([0, 1]); ax.set_yticks([0, 1]); ax.set_xticklabels(CLASSES); ax.set_yticklabels(CLASSES)
    ax.set_xlabel("predicted"); ax.set_ylabel("true"); ax.set_title(title, fontsize=9)
    plt.tight_layout()
    return fig

def train_one(name, step, seed, held_out):
    """One training; logs a W&B run. Returns (val_auc, held-out probabilities, epochs run, params)."""
    w = WIDTHS[(name, step)]
    tr = year != held_out
    Xtr, ytr, gtr = X[tr], y[tr], patient[tr]
    tr_i, va_i = next(GroupShuffleSplit(1, test_size=0.15, random_state=seed).split(Xtr, ytr, gtr))
    torch.manual_seed(seed); np.random.seed(seed)
    to_g = lambda a: torch.from_numpy(np.ascontiguousarray(a))[:, None].to(DEVICE)
    Xa, ya = to_g(Xtr[tr_i]), torch.from_numpy(ytr[tr_i].astype(np.float32)).to(DEVICE)
    Xv, yv = to_g(Xtr[va_i]), ytr[va_i]
    Xe = to_g(X[~tr]); ye = y[~tr]
    model = make_model(name, w).to(DEVICE)
    params = sum(p.numel() for p in model.parameters())
    scaler = torch.amp.GradScaler(enabled=DEVICE.type == "cuda")
    opt = torch.optim.AdamW(model.parameters(), lr=LR, weight_decay=WEIGHT_DECAY)
    pos_w = torch.tensor(float((ytr[tr_i] == 0).sum() / max((ytr[tr_i] == 1).sum(), 1)), device=DEVICE)
    state = {"best": -1.0, "bad": 0, "weights": None, "epochs": 0}
    run = wandb.init(project=WANDB_PROJECT, entity=WANDB_ENTITY or None, group=name, job_type="train", reinit=True,
                     name=f"{name}-{params / 1e3:.0f}k-fold{held_out}-s{seed}",
                     config=dict(model=name, step=step, target_M=LEVELS[step], width_mult=w, params=params, fold=held_out, seed=seed,
                                 lr=LR, weight_decay=WEIGHT_DECAY, batch=BATCH, max_epochs=MAX_EPOCHS, patience=PATIENCE))

    def on_epoch(ep, train_loss):
        pv = predict(model, Xv)
        auc = roc_auc_score(yv, pv)
        wandb.log({"epoch": ep + 1, "train_loss": train_loss, "val_auc": auc, "val_macro_f1": f1_score(yv, (pv >= THRESHOLD).astype(int), average="macro"),
                   "best_val_auc": max(state["best"], auc)})
        state["epochs"] = ep + 1
        if auc > state["best"] + 1e-4:
            state.update(best=auc, bad=0, weights={k: v.detach().clone() for k, v in model.state_dict().items()})
        else:
            state["bad"] += 1
        return state["bad"] >= PATIENCE

    t0 = time.time()
    run_epochs(model, opt, nn.BCEWithLogitsLoss(pos_weight=pos_w), Xa, ya, MAX_EPOCHS, scaler, on_epoch)
    model.load_state_dict(state["weights"])
    p = predict(model, Xe)
    m = metrics_of(ye, p)
    run.summary.update({**m, "val_auc_best": state["best"], "params": params, "epochs_run": state["epochs"], "train_seconds": time.time() - t0})
    fig = cm_figure(ye, p, f"{name} {params / 1e3:.0f}k · held-out {held_out} · seed {seed}")
    wandb.log({"confusion_matrix": wandb.Image(fig),
               "confusion_matrix_interactive": wandb.plot.confusion_matrix(y_true=[int(v) for v in ye], preds=[int(v) for v in (p >= THRESHOLD)], class_names=CLASSES)})
    plt.close(fig)
    wandb.finish()
    del model, Xa, ya, Xv, Xe
    if DEVICE.type == "cuda":
        torch.cuda.empty_cache()
    return state["best"], p, state["epochs"], params, m, time.time() - t0

runs = {}          # (model, step, fold) -> {"seeds": [{...}], "p": [...]}
y_fold = {f: y[year == f] for f in FOLDS}

In [ ]:
# ── Run: every model finishes all its sizes (large -> small) and folds before the next one starts; checkpoint after each (model, size, fold)
CKPT = OUT_DIR / "runs_checkpoint.pt"
signature = dict(PRESET=PRESET, FOLDS=FOLDS, SEEDS=SEEDS, MAX_EPOCHS=MAX_EPOCHS, LEVELS=LEVELS, MODELS=MODELS, LR=LR)
if CKPT.exists():
    saved = torch.load(CKPT, weights_only=False)
    if saved["signature"] == signature:
        runs = saved["runs"]; print(f"resumed {len(runs)} finished (model, size, fold) from {CKPT}")

fold_order = [OFFICIAL_FOLD] + [f for f in FOLDS if f != OFFICIAL_FOLD] if OFFICIAL_FOLD in FOLDS else list(FOLDS)
jobs = [(m, k, f) for m in MODELS for k in range(len(LEVELS)) for f in fold_order]
total, t_start, n_new, stopped, slowest = len(jobs), time.time(), 0, False, 0.0
for name, step, fold in jobs:
    key = (name, step, fold)
    if key in runs:
        continue
    if (time.time() - t_start) / 60 + slowest / 60 > TIME_BUDGET_MIN:
        stopped = True; break
    rec = {"seeds": [], "p": []}
    t_job = time.time()
    for seed in SEEDS:
        val_auc, p, epochs, params, m, secs = train_one(name, step, seed, fold)
        rec["seeds"].append(dict(seed=seed, val_auc=val_auc, epochs=epochs, params=params, seconds=secs, **m)); rec["p"].append(p)
    runs[key] = rec; n_new += 1
    slowest = max(slowest, time.time() - t_job)
    torch.save({"signature": signature, "runs": runs}, CKPT)
    eta = (time.time() - t_start) / n_new * (total - len(runs)) / 60
    pe = np.mean(rec["p"], 0)
    print(f"[{len(runs)}/{total}] {name:13s} {rec['seeds'][0]['params'] / 1e3:6.0f}k fold {fold} | val AUC {np.mean([s['val_auc'] for s in rec['seeds']]):.3f} | "
          f"held-out AUC {roc_auc_score(y_fold[fold], pe):.3f} macro-F1 {metrics_of(y_fold[fold], pe)['macro_f1']:.3f} | {time.time() - t_job:.0f} s | ~{eta:.0f} min left", flush=True)
if stopped:
    print(f"\nTime budget of {TIME_BUDGET_MIN} min reached: {len(runs)}/{total} finished. Run this cell again (resumes) or continue: the reports use what is finished.")

## 5 · Reports (CSV) and charts

In [ ]:
(OUT_DIR / "confusion_matrices").mkdir(exist_ok=True)
run_rows, cm_rows, ens_rows = [], [], []
for (name, step, fold), rec in runs.items():
    for s in rec["seeds"]:
        run_rows.append(dict(model=name, step=step, target_M=LEVELS[step], width_mult=WIDTHS[(name, step)], fold=fold, **s))
        cm_rows.append(dict(model=name, step=step, params=s["params"], fold=fold, level=f"seed {s['seed']}", **{k: s[k] for k in ("tn_W_as_W", "fp_W_as_FC", "fn_FC_as_W", "tp_FC_as_FC")}))
    pe = np.mean(rec["p"], 0)
    m = metrics_of(y_fold[fold], pe)
    params = rec["seeds"][0]["params"]
    ens_rows.append(dict(model=name, step=step, target_M=LEVELS[step], params=params, fold=fold, val_auc=np.mean([s["val_auc"] for s in rec["seeds"]]),
                         train_seconds=sum(s["seconds"] for s in rec["seeds"]), n_seeds=len(rec["seeds"]), **m))
    cm_rows.append(dict(model=name, step=step, params=params, fold=fold, level="ensemble", **{k: m[k] for k in ("tn_W_as_W", "fp_W_as_FC", "fn_FC_as_W", "tp_FC_as_FC")}))
per_run = pd.DataFrame(run_rows); ens = pd.DataFrame(ens_rows); cms = pd.DataFrame(cm_rows)
assert len(ens), "nothing finished: run the training cell"
per_run.to_csv(OUT_DIR / "report_per_run.csv", index=False)       # every single training (model, size, fold, seed)
cms.to_csv(OUT_DIR / "confusion_matrices.csv", index=False)       # confusion-matrix counts of every test (per seed and seed-ensemble)
ens.to_csv(OUT_DIR / "report_per_fold.csv", index=False)          # seed-ensemble per (model, size, fold)

# one confusion-matrix figure per model: rows = size, columns = held-out year
cm_files = []
for name in MODELS:
    steps = sorted({k for (m, k, f) in runs if m == name})
    folds = [f for f in FOLDS if any((name, k, f) in runs for k in steps)]
    if not steps or not folds:
        continue
    fig, axs = plt.subplots(len(steps), len(folds), figsize=(3.3 * len(folds), 3.1 * len(steps)), squeeze=False)
    for i, k in enumerate(steps):
        for j, f in enumerate(folds):
            ax = axs[i][j]
            if (name, k, f) not in runs:
                ax.axis("off"); continue
            rec = runs[(name, k, f)]; pe = np.mean(rec["p"], 0)
            cm = confusion_matrix(y_fold[f], (pe >= THRESHOLD).astype(int), labels=[0, 1]); cmn = cm / cm.sum(1, keepdims=True)
            ax.imshow(cmn, vmin=0, vmax=1, cmap="Blues")
            for a in range(2):
                for b in range(2):
                    ax.text(b, a, f"{cm[a, b]}\n{cmn[a, b]:.0%}", ha="center", va="center", fontsize=8, color="white" if cmn[a, b] > .5 else "black")
            ax.set_xticks([0, 1]); ax.set_yticks([0, 1]); ax.set_xticklabels(CLASSES); ax.set_yticklabels(CLASSES)
            ax.set_title(f"{rec['seeds'][0]['params'] / 1e3:.0f}k · {f}", fontsize=9)
    fig.suptitle(f"{name}: confusion matrices (rows true, columns predicted; seed ensemble)", y=1.0)
    plt.tight_layout(); path = OUT_DIR / "confusion_matrices" / f"cm_{name}.png"; plt.savefig(path, dpi=110); plt.show(); cm_files.append(path)

# summary: mean / std over the folds for each (model, size)
agg = ens.groupby(["model", "step"]).agg(target_M=("target_M", "first"), params=("params", "first"), n_folds=("fold", "nunique"),
        val_auc=("val_auc", "mean"), test_auc=("test_auc", "mean"), test_auc_std=("test_auc", "std"),
        macro_f1=("macro_f1", "mean"), macro_f1_std=("macro_f1", "std"), balanced_acc=("balanced_acc", "mean"),
        f1_W=("f1_W", "mean"), f1_FC=("f1_FC", "mean"), recall_W=("recall_W", "mean"), recall_FC=("recall_FC", "mean"),
        tn=("tn_W_as_W", "sum"), fp=("fp_W_as_FC", "sum"), fn=("fn_FC_as_W", "sum"), tp=("tp_FC_as_FC", "sum"),
        train_minutes=("train_seconds", lambda s: s.sum() / 60)).reset_index()
agg["model"] = pd.Categorical(agg["model"], MODELS, ordered=True)
agg = agg.sort_values(["model", "params"], ascending=[True, False]).reset_index(drop=True)
agg.to_csv(OUT_DIR / "report_summary.csv", index=False)
display(agg.round(4))

fig, axs = plt.subplots(1, 2, figsize=(12, 4.2))
for name in MODELS:
    d = agg[agg.model == name]
    for ax, col, lab in ((axs[0], "macro_f1", "held-out macro-F1"), (axs[1], "test_auc", "held-out AUC")):
        ax.errorbar(d["params"] / 1e6, d[col], yerr=d[col + ("_std" if col in ("macro_f1", "test_auc") else "")].fillna(0), marker="o", capsize=2, label=name)
        ax.set_xscale("log"); ax.set_xlabel("trainable parameters (M, log)"); ax.set_ylabel(lab + " (mean ± std over folds)"); ax.grid(alpha=.3)
axs[0].legend(); plt.tight_layout(); plt.savefig(OUT_DIR / "params_vs_performance.png", dpi=120); plt.show()

In [ ]:
# ── Final report (last step): recommended size of each model + overall, CSV; and a W&B summary run with charts and tables
final = agg.copy()
final["best_macro_f1_of_model"] = final.groupby("model", observed=True)["macro_f1"].transform("max")
final["within_tolerance"] = final["macro_f1"] >= final["best_macro_f1_of_model"] - TOLERANCE
final["recommended"] = False
for name in MODELS:
    ok = final[(final.model == name) & final.within_tolerance]
    if len(ok):
        final.loc[ok["params"].idxmin(), "recommended"] = True            # smallest size that is still within TOLERANCE of the model's best
final["rank_macro_f1"] = final["macro_f1"].rank(ascending=False, method="min").astype(int)
final["under_1M"] = final["params"] < 1_000_000
final.to_csv(OUT_DIR / "report_final.csv", index=False)
display(final[["model", "params", "macro_f1", "macro_f1_std", "test_auc", "balanced_acc", "recommended", "rank_macro_f1", "under_1M"]].round(4))
print("recommended sizes (smallest within", TOLERANCE, "macro-F1 of the model's best):")
print(final[final.recommended][["model", "params", "macro_f1", "test_auc"]].round(4).to_string(index=False))

summary_run = wandb.init(project=WANDB_PROJECT, entity=WANDB_ENTITY or None, name=f"summary-{PRESET}", job_type="summary", reinit=True,
                         config=dict(models=MODELS, levels_M=LEVELS, folds=FOLDS, seeds=SEEDS, preset=PRESET, tolerance=TOLERANCE))
wandb.log({"report_final": wandb.Table(dataframe=final.assign(model=final["model"].astype(str))),
           "report_per_run": wandb.Table(dataframe=per_run), "confusion_matrix_counts": wandb.Table(dataframe=cms),
           "params_vs_performance": wandb.Image(str(OUT_DIR / "params_vs_performance.png"))})
for path in cm_files:
    wandb.log({f"confusion_matrices/{path.stem}": wandb.Image(str(path))})
for _, r in final.iterrows():                                      # one point per size -> line charts "macro_f1 vs params" per model in the W&B UI
    wandb.log({f"macro_f1/{r['model']}": r["macro_f1"], f"test_auc/{r['model']}": r["test_auc"], f"params/{r['model']}": r["params"]})
for f in ("report_final.csv", "report_summary.csv", "report_per_run.csv", "confusion_matrices.csv"):
    wandb.save(str(OUT_DIR / f), base_path=str(OUT_DIR))
wandb.finish()

pack = Path("/content/gate2_param_reduction_results"); pack.mkdir(exist_ok=True)
for f in ("report_final.csv", "report_summary.csv", "report_per_fold.csv", "report_per_run.csv", "confusion_matrices.csv", "params_vs_performance.png"):
    shutil.copy2(OUT_DIR / f, pack / f)
shutil.copytree(OUT_DIR / "confusion_matrices", pack / "confusion_matrices", dirs_exist_ok=True)
zip_path = shutil.make_archive(str(pack), "zip", pack)
print("\nCSV reports in", OUT_DIR, "-> zip:", zip_path)

In [ ]:
# Optional: send the zip to YOUR computer (browser Colab; the VS Code extension shows a fallback link). Run only if you want the files locally.
import base64
from IPython.display import HTML, display
try:
    from google.colab import files
    files.download(zip_path)
except Exception as e:
    print("files.download not available here:", e)
    b64 = base64.b64encode(Path(zip_path).read_bytes()).decode()
    display(HTML(f'<a download="{Path(zip_path).name}" href="data:application/zip;base64,{b64}">Download {Path(zip_path).name}</a>'))